# **Anàlisi de Dades amb Python: Exercicis Pràctics Solucionats**

Benvinguts a la sessió pràctica del curs d'Anàlisi de Dades amb Python. En aquest notebook posarem en pràctica els conceptes d'**Exploració de Dades**, **Anàlisi de Supervivència** i **Clustering**.

---

### Instruccions
A cada apartat trobareu blocs de codi incomplets.  La vostra tasca consisteix a **omplir els espais en blanc representats per `____`** amb les funcions, mètodes o variables adequades per tal que el codi s'executi correctament.
Utilitzeu l'entorn per a probar, visualitzar resultats i practicar.

---

### Preparació de l'entorn (IMPORTANT)
**Executeu la cel·la de codi inferior** (fent clic al botó de "play") abans de començar amb els exercicis. Això instal·larà les llibreries que no vénen per defecte a Colab i carregarà les eines que emprarem durant la sessió.

In [ ]:
# Instal·lació de les llibreries
!pip install pandas numpy matplotlib seaborn lifelines scikit-learn

# Importació de les llibreries
# Dades i càlcul numèric (Bloc 2)
import pandas as pd
import numpy as np

# Visualització (Bloc 2)
import matplotlib.pyplot as plt
import seaborn as sns

# Anàlisi de supervivència
from lifelines import KaplanMeierFitter, CoxPHFitter
# Dataset
from lifelines.datasets import load_rossi

# Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


sns.set_theme(style="whitegrid")

# Carregar el dataset de Rossi
df = load_rossi()

# **Bloc 2: Exploració de dades amb Pandas**

En aquest bloc treballarem amb el dataset de Rossi, que ja es troba carregat i instanciat a la variable `df`.

Recorda que a Python, a diferència de l'R, apliquem **mètodes** directament sobre l'objecte (per exemple, `objecte.metode()`) en lloc d'envoltar l'objecte amb funcions (diapositiva 11).

**Exercici 2.1: Inspecció inicial i estadístiques bàsiques**

L'objectiu d'aquest exercici és conèixer el conjunt de dades. Volem obtenir un resum estadístic (l'equivalent a `summary()` de l'R) i veure quants valors únics hi ha en una columna específica (l'equivalent a `table()`).

**Tasques a fer:**
1. Utilitza el mètode adequat per mostrar les estadístiques descriptives de tot el dataframe (mitjana, mediana, màxim, etc.).
2. Utilitza el mètode adequat per comptar quants individus han estat arrestats i quants no (variable `arrest`).

Pista: Podeu trobar més informació a la diapositiva 17.

In [ ]:
# 1. Mostra les estadístiques descriptives de tot el dataframe
estadistiques = df.describe()
print("Estadístiques descriptives:")
print(estadistiques)

# 2. Compta quants individus tenim a cada categoria de la variable 'arrest' (1 = Sí, 0 = No)
reincidencies = df['arrest'].value_counts()
print("\nReincidències:")
print(reincidencies)

**Exercici 2.2: Indexació i Filtratge**

A Pandas, podem filtrar files passant condicions booleanes dins dels claudàtors `[]`, de manera molt similar a com ho faríem amb `filter()` de `dplyr` a R.

**Tasques a fer:**
1. Crea un subconjunt de dades només amb aquells individus que **sí** van rebre ajuda financera (`fin == 1`) **i** que tenen una edat (`age`) superior a 25 anys.
2. Mostra només les primeres 3 files d'aquest nou dataframe filtrat.

Pista: Podeu trobar més informació a la diapositiva 18.

In [ ]:
# 1. Filtra els individus amb ajuda financera (fin == 1) i edat superior a 25
df_filtrat = df[(df['fin'] == 1) & (df['age'] > 25)]

# 2. Mostra les primeres 3 files del resultat
df_filtrat.head(3)

# **Bloc 3: Anàlisi de Supervivència**

En aquest bloc continuem treballant amb el dataset de Rossi, que es troba instanciat a la variable `df`.

**Exercici 3.1: Corba de Kaplan-Meier General**

L'objectiu de l'exercici és crear una corba de supervivència per a l'estudi de Rossi. L'esdeveniment que estudiem és la reincidència criminal (`arrest`), i el temps de seguiment es mesura en setmanes (`week`).

**Tasques a fer:**
1. Ajusta l'estimador amb la columna que representa el temps (durada fins a l'esdeveniment o censura) i la columna que indica si l'esdeveniment ha ocorregut.
2. Dibuixa la funció de supervivència.

Pista: Podeu trobar més informació a la diapositiva 27.

In [ ]:
# 1. Instanciem l'estimador de Kaplan-Meier
kmf = KaplanMeierFitter()

# 2. Ajustem l'estimador amb les dades del dataframe
kmf.fit(durations=df['week'], event_observed=df['arrest'])

# 3. Dibuixem la corba de supervivència
kmf.plot_survival_function()

# Afegim títol i etiquetes
plt.title('Corba de Supervivència de Rossi')
plt.xlabel('Setmanes')
plt.ylabel('Probabilitat de Supervivència (No reincidir)')
plt.show()

**Exercici 3.2: Comparació de Corbes per Grups**

A l'estudi de Rossi, l'objectiu principal era veure si donar ajuda financera (`fin`) reduïa la reincidència. Per fer-ho visualment, podem sobreposar les dues corbes de Kaplan-Meier en un mateix gràfic.

**Tasques a fer:**
1. Utilitza la funció d'agrupació de Pandas per iterar sobre els grups creats per la variable `fin`.
2. Indica a la funció de dibuix que utilitzi els eixos del gràfic (`ax`) per tal que les dues corbes es sobreposin i no es dibuixin en gràfics separats.

Pista: Podeu trobar més informació a la diapositiva 29.

In [ ]:
# Creem la figura i els eixos sobre els quals dibuixarem
fig, ax = plt.subplots(figsize=(8,5))

# 1. Agrupem el dataframe per la variable 'fin' i iterem per cada grup
for nom_grup, df_grup in df.groupby('fin'):
    kmf_grup = KaplanMeierFitter()

    # Ajustem l'estimador per a cada subgrup
    # Etiquetem la corba amb el nom del grup (0 = Sense ajuda, 1 = Amb ajuda)
    kmf_grup.fit(durations=df_grup['week'], event_observed=df_grup['arrest'], label=f"Grup fin={nom_grup}")

    # 2. Dibuixem la corba obligant-la a pintar-se al mateix eix
    kmf_grup.plot_survival_function(ax=ax)

# Afegim titol i etiquetes
plt.title('Supervivència segons ajut financer')
plt.xlabel('Temps (setmanes)')
plt.ylabel('Probabilitat de Supervivència')
plt.grid(alpha=0.3)
plt.show()

# **Bloc 4: Clustering**

En aquest cas, usarem el dataset de "Wine". Per tal de carregar-ho i instanciar-ho, cal que executeu la següent cel·la (no cal modificar res).

Per fer ús del dataset ja escalat i preparat, caldrà usar la variable `X_scaled`.

In [ ]:
# Importem el dataset de vins i l'eina d'escalat
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

# Carreguem el dataset
wine_data = load_wine()
X = wine_data.data

# Instanciem l'escalador i transformem les dades
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

**Exercici 4.1: Trobar el nombre òptim de clústers (Mètode del Colze)**

Per a poder usar K-Means, cal especificar el nombre de `clusters`. Tal com hem vist, per a trobar el nombre òptim de clusters *k*, utilitzarem el "Mètode del Colze". Busquem el punt on afegir més grups ja no redueix substancialment la inèrcia.

**Tasques a fer:**
1. A l'hora d'instanciar el model dins del bucle, assigna el nombre de clústers perquè sigui igual a la variable del bucle iteratiu. És a dir, assigna al nombre de clusters el valor del comptador emprat en el bucle.
2. Guarda la mètrica d'inèrcia (suma de distàncies intra-clúster). Recorda que no cal calcular-la, s'autocalcula després d'ajustar el model.

Pista: Podeu trobar més informació a la diapositiva 42.

In [ ]:
inercies = []
valors_k = range(1, 10)

for k in valors_k:
    # 1. Instanciem el KMeans indicant el nombre de clústers dinàmic
    km = KMeans(n_clusters=k, random_state=42, n_init=10)

    # Ajustem a les dades escalades
    km.fit(X_scaled)

    # 2. Guardem la inèrcia
    inercies.append(km.inertia_)

# Dibuixem el gràfic del colze (eix x -> valors_k, eix y -> inercies)
plt.figure(figsize=(8,5))
plt.plot(valors_k, inercies, marker='o', color='blue', linewidth=2)
plt.xlabel('Nombre de Clústers (k)')
plt.ylabel('Inèrcia (Distància Intra-cluster)')
plt.title('Mètode del Colze - Wine Dataset')
plt.grid(alpha=0.3)
plt.show()

**Exercici 4.2: Visualització Final dels Clústers**

En executar la cel·la anterior, hauràs observat que el "colze" es forma a $k=3$. Ara entrenarem el model definitiu amb aquest valor i dibuixarem els resultats. Com que tenim 13 variables, visualitzarem només les dues primeres columnes per poder fer un gràfic 2D.

**Tasques a fer:**
1. Pinta els punts del gràfic de dispersió assignant el color segons l'etiqueta del clúster a la qual pertany cadascun.
2. Dibuixa els centroides (els "centres" de cada grup) accedint a l'atribut corresponent del model.

In [ ]:
# Entrenem el model definitiu amb el k òptim (k = 3, com ja hem vist)
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_final.fit(X_scaled)

fig, ax = plt.subplots(figsize=(8,5))

# 1. Dibuixem els punts donant color segons l'etiqueta del clúster
# X_scaled[:, 0] i X_scaled[:, 1] agafen la primera i segona variable respectivament
scatter = ax.scatter(X_scaled[:, 0], X_scaled[:, 1],
                     c=kmeans_final.labels_, cmap='viridis', alpha=0.6, s=50)

# 2. Dibuixem els centroides damunt del gràfic en color vermell
centroides = kmeans_final.cluster_centers_
ax.scatter(centroides[:, 0], centroides[:, 1],
           c='red', marker='X', s=200, edgecolors='black', linewidths=2, label='Centroides')

# Afegim titol i etiquetes
plt.colorbar(scatter, label='Clúster assignat')
plt.title('K-Means Clustering (k=3)')
plt.legend()
plt.show()